# Causal saliency — full sweep (v4.3 hybrid, **STUB**)

Sweep notebook that runs every attribution primitive across a stratified
sample of in-vocab sequences and writes one Parquet row + one `.npz` payload
per sequence to disk for downstream analysis.

**DO NOT RUN ON v4.3** — the v4.3 split has species leakage. This notebook
is plumbing only; swap `CKPT` to a properly-split checkpoint before any
serious sweep. The smoke notebook (`causal_saliency_smoke.ipynb`) is what
proves the primitives work.

## Design decisions
- **One row per sequence** in `index.parquet` (header, sf, class, tir,
  rho_N, rho_shuffle, rho_reverse, deletion_auc_saliency,
  deletion_auc_random, deletion_auc_gap, suff_max_logit,
  cross_modal_drop_cnn, cross_modal_drop_gnn, n_windows, seq_len, runtime_s).
- **Heavy arrays** (saliency, IG, occlusion drops, suff curve, deletion
  curve) go to `arrays/<header_safe>.npz` so we can re-aggregate later
  without re-running the model.
- **Resumable**: each sequence checks for its `.npz` before computing; if
  present and `--force` not set, it is skipped. The index Parquet is
  rebuilt at the end from whatever `.npz` files exist.
- **Stratified sample**: ≤MAX_PER_SF sequences per superfamily, sampled with
  fixed seed, so smaller superfamilies are not drowned out.

In [ ]:
import os, sys, json, time, hashlib, gc
import numpy as np
import pandas as pd
import torch

def _find_repo(start: str) -> str:
    p = os.path.abspath(start)
    for _ in range(6):
        if os.path.isdir(os.path.join(p, "data", "vgp")):
            return p
        p = os.path.dirname(p)
    raise RuntimeError(f"could not locate repo root from {start}")
REPO = _find_repo(os.getcwd())
NB_DIR = os.path.join(REPO, "model_result_interp")
if NB_DIR not in sys.path:
    sys.path.insert(0, NB_DIR)
import causal_saliency_hybrid as csh

# --- config ----------------------------------------------------------------
CKPT = os.path.join(REPO, "data_analysis", "vgp_model_data_tpase_multi", "v4.3",
                    "hybrid_v4.3_epoch40.pt")  # SWAP for fixed-split checkpoint
LABELS_TSV = os.path.join(REPO, "model_result_interp", "interpretation_results",
                          "causal_saliency_hybrid", "labels.tsv")
FASTA = os.path.join(REPO, "data", "vgp", "all_vgp_tes.fa")

OUT_DIR = os.path.join(REPO, "model_result_interp", "interpretation_results",
                       "causal_saliency_hybrid", "sweep_v4.3")
ARR_DIR = os.path.join(OUT_DIR, "arrays")
os.makedirs(ARR_DIR, exist_ok=True)

MAX_PER_SF = 200          # stratified cap per superfamily
SEED = 1234
OCC_WINDOW = 200
OCC_STRIDE = 200
SUFF_WINDOW = 600
SUFF_STRIDE = 300
IG_STEPS = 8
DEL_STEPS = 12
FORCE = False             # re-run even if .npz exists
LIMIT = None              # cap total sequences (None = all). Set small int for dry runs.

device = csh.resolve_device()
print("device:", device, "| out:", OUT_DIR)

## 1. Load model + label table, build stratified sample

In [ ]:
import csv
model, class_names, sf_names = csh.load_hybrid_checkpoint(CKPT, device)
ckpt_sha = hashlib.sha256(open(CKPT, "rb").read()).hexdigest()[:12]
print("checkpoint sha:", ckpt_sha)

rows = list(csv.DictReader(open(LABELS_TSV), delimiter="\t"))
usable = [r for r in rows if r["in_model_vocab"] == "1"]
print(f"usable rows: {len(usable)}")

# Stratified sample
rng = np.random.default_rng(SEED)
by_sf: dict[str, list] = {}
for r in usable:
    by_sf.setdefault(r["superfamily"], []).append(r)
sampled = []
for sf, lst in by_sf.items():
    idx = rng.permutation(len(lst))[:MAX_PER_SF]
    sampled.extend([lst[i] for i in idx])
rng.shuffle(sampled)
if LIMIT:
    sampled = sampled[:LIMIT]
print(f"stratified sample: {len(sampled)} sequences across {len(by_sf)} superfamilies")

## 2. Per-sequence helper

Computes every primitive for one sequence and writes one `.npz`. Returns the
row dict for the index. Idempotent: skips if `.npz` already exists and not
`FORCE`.

In [ ]:
def _safe_name(header: str) -> str:
    return header.replace("/", "_").replace("#", "_").replace(":", "_")[:120]

def _load_seq(meta) -> str:
    with open(FASTA, "rb") as f:
        f.seek(int(meta["fa_offset"]))
        chunk = f.read(int(meta["fa_byte_len"])).decode("ascii")
    return "".join(chunk.split("\n")[1:]).upper()

featurizer = csh.KmerFeaturizer()

def _cross_modal_drop(enc, target_sf):
    """Ablate each tower (zero its contribution) and measure logit drop.
    NOTE: stub — depends on csh exposing forward hooks for tower outputs.
    Until then, returns (np.nan, np.nan) and the analysis notebook skips it.
    """
    return float("nan"), float("nan")

def process_one(meta) -> dict | None:
    header = meta["header"]
    npz_path = os.path.join(ARR_DIR, _safe_name(header) + ".npz")
    if os.path.exists(npz_path) and not FORCE:
        return None  # already done; index rebuilt at end
    t0 = time.time()
    seq = _load_seq(meta)
    if len(seq) < 50:
        return None
    enc = csh.encode_sequence(seq, header)

    # forward to choose target = predicted sf
    sf_logits = csh.predict_logits(model, [enc], featurizer, device,
                                   batch_size=1, head=csh.HEAD_SUPERFAMILY)
    target_sf = int(sf_logits.argmax(axis=1)[0])

    sal = csh.compute_saliency(model, enc, target_sf, featurizer, device,
                               head=csh.HEAD_SUPERFAMILY)
    ig = csh.compute_integrated_gradients(model, enc, target_sf, featurizer, device,
                                          steps=IG_STEPS, head=csh.HEAD_SUPERFAMILY)
    occN = csh.occlusion_profile(model, enc, [target_sf], featurizer, device,
                                 window=OCC_WINDOW, stride=OCC_STRIDE, mode="N",
                                 batch_size=8, head=csh.HEAD_SUPERFAMILY)
    occS = csh.occlusion_profile(model, enc, [target_sf], featurizer, device,
                                 window=OCC_WINDOW, stride=OCC_STRIDE, mode="shuffle",
                                 batch_size=8, head=csh.HEAD_SUPERFAMILY)
    occR = csh.occlusion_profile(model, enc, [target_sf], featurizer, device,
                                 window=OCC_WINDOW, stride=OCC_STRIDE, mode="reverse",
                                 batch_size=8, head=csh.HEAD_SUPERFAMILY)
    suf = csh.keep_only_window_profile(model, enc, [target_sf], featurizer, device,
                                       window=SUFF_WINDOW, stride=SUFF_STRIDE,
                                       batch_size=8, head=csh.HEAD_SUPERFAMILY)
    dc = csh.deletion_curve(model, enc, sal, target_sf, featurizer, device,
                            n_steps=DEL_STEPS, head=csh.HEAD_SUPERFAMILY)
    cm_cnn, cm_gnn = _cross_modal_drop(enc, target_sf)

    np.savez_compressed(
        npz_path,
        sal=sal.astype(np.float32),
        ig=ig.astype(np.float32),
        occN_drops=occN["drops"][0].astype(np.float32),
        occN_centers=occN["centers"].astype(np.int32),
        occS_drops=occS["drops"][0].astype(np.float32),
        occR_drops=occR["drops"][0].astype(np.float32),
        suf=suf["survived"][0].astype(np.float32),
        suf_centers=suf["centers"].astype(np.int32),
        del_saliency=dc["curve_saliency"].astype(np.float32),
        del_random=dc["curve_random"].astype(np.float32),
    )

    return dict(
        header=header,
        genome=meta["genome"],
        cls=meta["class"],
        superfamily=meta["superfamily"],
        target_sf_id=target_sf,
        target_sf_name=sf_names[target_sf],
        tir=int(meta["tir"]) if meta["tir"] in ("0", "1") else -1,
        seq_len=int(meta["seq_len"]),
        n_windows=int(occN["drops"].shape[1]),
        rho_N=csh.saliency_occlusion_correlation(sal, occN),
        rho_shuffle=csh.saliency_occlusion_correlation(sal, occS),
        rho_reverse=csh.saliency_occlusion_correlation(sal, occR),
        suff_max_logit=float(np.max(suf["survived"][0])),
        deletion_auc_saliency=float(dc["auc_saliency"]),
        deletion_auc_random=float(dc["auc_random"]),
        deletion_auc_gap=float(dc["auc_gap"]),
        cross_modal_drop_cnn=cm_cnn,
        cross_modal_drop_gnn=cm_gnn,
        runtime_s=time.time() - t0,
        ckpt_sha=ckpt_sha,
    )

## 3. Run the sweep (resumable)

This is the long cell. It writes `<header>.npz` per sequence and accumulates
rows in memory. If the kernel dies, just re-run — already-done sequences are
skipped because their `.npz` exists.

**For a dry run**, set `LIMIT = 5` in cell 2 and re-run from cell 2.

In [ ]:
new_rows = []
errors = []
t0 = time.time()
for i, meta in enumerate(sampled):
    try:
        row = process_one(meta)
        if row is not None:
            new_rows.append(row)
    except Exception as e:
        errors.append(dict(header=meta["header"], err=repr(e)))
    if (i + 1) % 25 == 0:
        elapsed = time.time() - t0
        rate = (i + 1) / elapsed
        eta = (len(sampled) - i - 1) / rate if rate > 0 else float("inf")
        print(f"  [{i+1}/{len(sampled)}] {rate:.2f} seq/s, ETA {eta/60:.1f} min, "
              f"new={len(new_rows)}, errs={len(errors)}")
        gc.collect()
print(f"DONE — new_rows={len(new_rows)}, errors={len(errors)}, "
      f"total time {(time.time()-t0)/60:.1f} min")
if errors:
    print("first 5 errors:")
    for e in errors[:5]:
        print(" ", e)

## 4. Rebuild `index.parquet` from disk

We rebuild from disk so that resumed runs include previously-computed
sequences too.

In [ ]:
# Rebuild index by walking arrays/ — but we only have the npz arrays, not the
# scalar summary. So in this stub we just write the rows we computed in cell 3.
# A future fuller version could re-derive scalars from the npz on disk.
INDEX_PATH = os.path.join(OUT_DIR, "index.parquet")
if new_rows:
    df_new = pd.DataFrame(new_rows)
    if os.path.exists(INDEX_PATH):
        df_old = pd.read_parquet(INDEX_PATH)
        df = pd.concat([df_old[~df_old["header"].isin(df_new["header"])], df_new],
                       ignore_index=True)
    else:
        df = df_new
    df.to_parquet(INDEX_PATH, index=False)
    print(f"wrote {INDEX_PATH} with {len(df)} rows")
else:
    print("no new rows; index.parquet unchanged")

# Save run manifest
manifest = dict(
    ckpt=CKPT, ckpt_sha=ckpt_sha, seed=SEED, max_per_sf=MAX_PER_SF,
    occ_window=OCC_WINDOW, occ_stride=OCC_STRIDE,
    suff_window=SUFF_WINDOW, suff_stride=SUFF_STRIDE,
    ig_steps=IG_STEPS, del_steps=DEL_STEPS,
    n_sampled=len(sampled), n_completed=len(new_rows),
    errors=errors[:50],
)
with open(os.path.join(OUT_DIR, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)
print("manifest written")

## TODO before this is "real"

- [ ] Implement `_cross_modal_drop` (cell 5 placeholder) — needs hooks
      inside `csh.HybridV43` to zero each tower's contribution at fusion
      time. Currently returns NaN.
- [ ] Re-derive scalar summaries from `.npz` files when rebuilding the index
      (so you can change the rho metric without re-running everything).
- [ ] Add a **random-init negative-control sweep** in a sibling notebook
      (`causal_saliency_negctrl.ipynb`) that runs the same primitives with a
      freshly-initialised `HybridV43` to confirm rho collapses.
- [ ] Switch `CKPT` to the fixed-split checkpoint when it exists.